In [1]:
# Computes stochastic control closure certificate using SOS for stochastic systems in verifying LTL specifications
# Becoming Reach Once Example

# include important libraries
using JuMP
using MosekTools
using DynamicPolynomials
using MultivariatePolynomials
using LinearAlgebra
using TSSOS # important for SOS, see https://github.com/wangjie212/TSSOS

@polyvar x y x0 # global vars used in monomials
var = [x, y]
sos_tol = 1 # the maximum degree of unknown SOS lagrange polynomials = deg + sos_tol 
error = 2   # precision digit places
gamma, lamda, delta, tau1, tau2, tau3 = 0.85, 100, 0.5, 0.2, 0.06, 0.5 # SC3 constants and S-procedure constants

U = [-0.25, -0.2, -0.15, -0.1, -0.05, 0.0, 0.05, 0.1, 0.15, 0.2, 0.25] # control inputs
rho = 1/length(U) # uniform control bilinearity coefficients

# returns 1 with 51% chance, -1 with 49% chance
wp, wm = 1, -1

# access odd priorities
function odd_pri(d::Dict)
    return [v for v in values(d) if isodd(v)]
end

# creates the parametrized SCC
function add_paramC_poly!(model, var1, deg, q, p)
    basis = monomials(var1, 0:deg) # basis in variables
    coeffs = @variable(model, [1:length(basis)], base_name="c_$(q)_$(p)")
    C_qp = sum(coeffs[i] * basis[i] for i in 1:length(basis))
    return C_qp, coeffs, basis
end

function add_paramCf_poly!(model, var1, deg, q, p)
    basis = monomials(var1, 0:deg) # basis in variables
    coeffs = @variable(model, [1:length(basis)], base_name="cf_$(q)_$(p)")
    Cf_qp = sum(coeffs[i] * basis[i] for i in 1:length(basis))
    return Cf_qp, coeffs, basis
end

# creates the parametrized ranking functions
function add_paramV_poly!(model, var2, deg, l, q, p)
    basis = monomials(var2, 0:deg)
    coeffs = @variable(model, [1:length(basis)], base_name="v_$(l)_$(q)_$(p)")
    V_l_qp = sum(coeffs[i] * basis[i] for i in 1:length(basis))
    return V_l_qp, coeffs, basis
end

# the domain is function pairwise
# X0=[50]
gX = [x-0, y-0]
g5 = [x0-50, 50-x0, x-0, y-0]

# polynomial stochastic closure certificate of degree deg
function scc(deg, S, pri)
    # synthesize SCC by using the standard formulation
    # deg: degree of scc template
    model = Model(optimizer_with_attributes(Mosek.Optimizer))
    set_optimizer_attribute(model, MOI.Silent(), true)
    
    C_dict = Dict()  # key => (symbolic_poly, numeric_poly)
    for q in keys(S1) # DPA states
        for (lab, qn) in S[q]
            C, Cc, Cb = add_paramC_poly!(model, var, deg, q, qn) # CC, CC coefficients, and CC basis functions
            Cf, Ccf, Cbf = add_paramCf_poly!(model, var, deg, q, qn)
            key_c = (q, qn, lab, "c") 
            key_cf = (q, qn, lab, "cf")
            C_dict[key_c] = (C, Cc, Cb)  # store symbolic, coefficients, basis
            C_dict[key_cf] = (Cf, Ccf, Cbf)

            # cond 1
            C_plus1 = C([x, y]=>[x, x + wp])
            C_minus1 = C([x, y]=>[x, x - wm])
            E_C1 = sum(rho * ((0.5 + U[i]) * C_plus1 + (0.5 - U[i]) * C_minus1) for i in eachindex(U))
            add_psatz!(model, -E_C1 + gamma, [x], [x-0], [], div(deg+sos_tol,2), QUIET=true, CS=false, TS=false, GroebnerBasis=true) 

            for u in U
                C_plus2to4 = C([x, y]=>[x, x + wp])
                C_minus2to4 = C([x, y]=>[x, x - wm])
                E_C2to4 = (0.5 + u) * C_plus2to4 + (0.5 - u) * C_minus2to4
                # cond 2
                if pri[q] <= pri[qn]
                    C_plus2 = Cf([x, y]=>[x, x + wp])
                    C_minus2 = Cf([x, y]=>[x, x - wm])
                    E_Cf2 = (0.5 + u)*C_plus2 + (0.5 - u)*C_minus2
                    add_psatz!(model, -E_Cf2 + gamma - tau3 * (lamda - E_C2to4), [x], [x-0], [], div(deg+sos_tol,2), QUIET=true, CS=false, TS=false, GroebnerBasis=true) 
                end

                for p in keys(S1)
                    # cond 3
                    Cf3e, Ccf3e, Cbf3e = add_paramCf_poly!(model, var, deg, p, qn) # e as in E[.\.]
                    Cf3, Ccf3, Cbf3 = add_paramCf_poly!(model, var, deg, p, q) 
                    key_cf3 = (p, q, "cf") 
                    key_cf3e = (p, qn, "cf")
                    C_dict[key_cf3] = (Cf3, Ccf3, Cbf3)  
                    C_dict[key_cf3e] = (Cf3e, Ccf3e, Cbf3e)
                    if pri[p] <= pri[q] && pri[p] <= pri[qn] 
                        Cf_plus3 = Cf3e([x, y]=>[x, y + wp])
                        Cf_minus3 = Cf3e([x, y]=>[x, y - wm])
                        E_Cf3 = (0.5 + u)*Cf_plus3 + (0.5 - u)*Cf_minus3    
                        add_psatz!(model, Cf3 - E_Cf3 - tau3 * (lamda - E_C2to4), [x, y], [x-0, y-0], [], div(deg+sos_tol,2), QUIET=true, CS=false, TS=false, GroebnerBasis=true)              
                    end

                    # cond 4
                    C4e, Cc4e, Cb4e = add_paramC_poly!(model, var, deg, p, qn) # e as in E[.\.]
                    C4, Cc4, Cb4 = add_paramC_poly!(model, var, deg, p, q) 
                    key_c4 = (p, q, "c") 
                    key_c4e = (p, qn, "c")
                    C_dict[key_c4] = (C4, Cc4, Cb4)  
                    C_dict[key_c4e] = (C4e, Cc4e, Cb4e)
                    C_plus4 = C4e([x, y]=>[x, y + wp])
                    C_minus4 = C4e([x, y]=>[x, y - wm])
                    E_C4 = (0.5 + u)*C_plus4 + (0.5 - u)*C_minus4
                    add_psatz!(model, C4 - E_C4 - tau3 * (lamda - E_C2to4), [x, y], [x-0, y-0], [], div(deg+sos_tol,2), QUIET=true, CS=false, TS=false, GroebnerBasis=true) 
    
                    #### non-negativity of C and Cf
                    add_psatz!(model, C4, var, gX, [], div(deg+sos_tol,2), QUIET=true, CS=false, TS=false, GroebnerBasis=true) 
                    add_psatz!(model, Cf3, var, gX, [], div(deg+sos_tol,2), QUIET=true, CS=false, TS=false, GroebnerBasis=true)
                
                    # cond 5
                    for l in odd_pri(pri)
                        V2, Vc2, Vb2 = add_paramV_poly!(model, [x0, y], deg, l, "q0", q)
                        V1, Vc1, Vb1 = add_paramV_poly!(model, [x0, x], deg, l, "q0", p)
                        key_v1 = (l, "q0", p, "v")
                        key_v2 = (l, "q0", q, "v")
                        C_dict[key_v1], C_dict[key_v2] = (V1, Vc1, Vb1), (V2, Vc2, Vb2)
                        #### non-negativity of V
                        add_psatz!(model, V1, var, gX, [], div(deg+sos_tol,2), QUIET=true, CS=false, TS=false, GroebnerBasis=true)
                        add_psatz!(model, V2, var, gX, [], div(deg+sos_tol,2), QUIET=true, CS=false, TS=false, GroebnerBasis=true)
                
                        C1 = C4([x, y]=>[x0, x])
                        if l <= pri[p] && pri[p] <= pri[q]
                            add_psatz!(model, -V2 + V1 - delta + tau1*(C1 - lamda) + tau2*(Cf3 - lamda), [x0,x,y], g5, [], div(deg+sos_tol,2), QUIET=true, CS=false, TS=false, GroebnerBasis=true)
                        end
                    end
                end
            end
        end
    end
        
    optimize!(model) # solve for coefficients
    status = termination_status(model)
    C_eval_dict = Dict() # Get numerical values of coefficients and plug into polynomials
    for (key, (C, Cc, Cb)) in C_dict
        coeff_vals = round.(value.(Cc); digits=error)  # Round each coefficient to 2 decimal places
        C_numeric = sum(coeff_vals[i] * Cb[i] for i in eachindex(Cb))
        C_eval_dict[key] = (C, C_numeric)
    end
    
    # status might be optimal but if all Bc approx 10^{-error}, it's essentially 0 so OPTIMAL != CC.
    p_sat = 1-gamma/lamda # satisfaction probability lower bound
    return status, C_eval_dict, p_sat
end

# the considered DPA 

# A representing ~aUb
S1 = Dict(
    :"q0" => [("b", :"q1"), ("c", :"q0"), ("a", :"q2")],
    :"q1" => [("a", :"q1"), ("b", :"q1"), ("c", :"q1")],
    :"q2" => [("a", :"q2"), ("b", :"q2"), ("c", :"q2")]
)

pri1 = Dict(:"q0" => 3, :"q1" => 2, :"q2" => 3) # priority function


# Simulation
deg = 1 # desired degree of SCC
stats = @timed (status, CC_data, p_sat) = scc(deg, S1, pri1)

println("poly deg: ", deg)
println("status: ", status)
println("sat probability: ", p_sat)
println("Number of SCC polynomials: ", length(CC_data))
println("time: ", stats.time, "\n")

for (k, v) in CC_data
    println(k, ": ", v[2], ",")
end

println("Finished")

poly deg: 1
status: OPTIMAL
sat probability: 0.9915
Number of SCC polynomials: 39
time: 11.178993583

("q0", "q1", "b", "cf"): 0.0,
("q1", "q1", "c", "cf"): 45.55 - 26.51*y - 72.07*x,
("q2", "q1", "cf"): 0.0,
("q0", "q1", "cf"): 0.0,
("q0", "q1", "c"): 69.92 - 2.79*y + 115.48*x,
("q1", "q1", "a", "cf"): 45.55 - 26.51*y - 72.07*x,
("q2", "q1", "c"): 69.92 - 2.79*y + 115.48*x,
("q1", "q1", "c", "c"): 4.38 - 43.41*y - 47.79*x,
("q1", "q2", "c"): 72.77 + 0.41*y + 110.0*x,
("q1", "q2", "cf"): 72.77 + 0.41*y + 110.0*x,
(3, "q0", "q2", "v"): 20.39 + 51.88*x0 + 9.63*y,
("q0", "q1", "b", "c"): 4.23 - 43.27*y - 47.5*x,
("q2", "q2", "cf"): -167.37 + 244.87*y + 109.07*x,
("q2", "q0", "c"): 70.72 + 68.79*y + 70.13*x,
("q1", "q0", "c"): 70.75 + 71.9*y + 70.19*x,
("q2", "q0", "cf"): 72.85 + 321.09*y + 69.26*x,
("q0", "q2", "c"): 75.83 - 2.68*y + 109.94*x,
("q2", "q2", "c", "cf"): 51.55 - 26.39*y - 77.95*x,
(3, "q0", "q0", "v"): 69.88 + 38.4*x0 + 63.94*x,
("q1", "q1", "a", "c"): 4.38 - 43.41*y - 47.79

#### To post verify the SOS SC3 via Z3 and remember to change the kernel to python

In [43]:
# SC^2 for Gambler's ruin without DPA

from z3 import *
import itertools
import numpy as np
import time
from itertools import product

# --- define symbolic polynomial variables ---
x0, x, y = Reals('x0 x y')

############################ Dynamics ###############
# SC2 from SOS
D = {
 ("q0", "q1", "b", "cf"): 0.0,
("q1", "q1", "c", "cf"): 45.55 - 26.51*y - 72.07*x,
("q2", "q1", "cf"): 0.0,
("q0", "q1", "cf"): 0.0,
("q0", "q1", "c"): 69.92 - 2.79*y + 115.48*x,
("q1", "q1", "a", "cf"): 45.55 - 26.51*y - 72.07*x,
("q2", "q1", "c"): 69.92 - 2.79*y + 115.48*x,
# ("q1", "q1", "c", "c"): 4.38 - 43.41*y - 47.79*x,
("q1", "q2", "c"): 72.77 + 0.41*y + 110.0*x,
("q1", "q2", "cf"): 72.77 + 0.41*y + 110.0*x,
(3, "q0", "q2", "v"): 20.39 + 51.88*x0 + 9.63*y,
# ("q0", "q1", "b", "c"): 4.23 - 43.27*y - 47.5*x,
("q2", "q2", "cf"): -167.37 + 244.87*y + 109.07*x,
("q2", "q0", "c"): 70.72 + 68.79*y + 70.13*x,
("q1", "q0", "c"): 70.75 + 71.9*y + 70.19*x,
("q2", "q0", "cf"): 72.85 + 321.09*y + 69.26*x,
("q0", "q2", "c"): 75.83 - 2.68*y + 109.94*x,
("q2", "q2", "c", "cf"): 51.55 - 26.39*y - 77.95*x,
(3, "q0", "q0", "v"): 69.88 + 38.4*x0 + 63.94*x,
# ("q1", "q1", "a", "c"): 4.38 - 43.41*y - 47.79*x,
("q0", "q2", "cf"): -167.37 + 244.87*y + 109.07*x,
("q1", "q1", "b", "cf"): 45.55 - 26.51*y - 72.07*x,
("q0", "q0", "c", "cf"): 51.55 - 26.39*y - 77.95*x,
("q1", "q1", "cf"): 66.86 + 0.3*y + 115.53*x,
("q0", "q2", "a", "cf"): 51.55 - 26.39*y - 77.95*x,
("q2", "q2", "b", "cf"): 51.55 - 26.39*y - 77.95*x,
# ("q2", "q2", "c", "c"): -7.66 - 43.64*y - 35.98*x,
# ("q2", "q2", "a", "c"): -7.66 - 43.64*y - 35.98*x,
("q2", "q2", "c"): 75.83 - 2.68*y + 109.94*x,
("q0", "q0", "c"): 70.72 + 68.79*y + 70.13*x,
# ("q1", "q1", "b", "c"): 4.38 - 43.41*y - 47.79*x,
# ("q1", "q0", "cf"): 70.75 + 71.9*y + 70.19*x,
# ("q2", "q2", "b", "c"): -7.66 - 43.64*y - 35.98*x,
# ("q0", "q0", "c", "c"): -7.66 - 43.64*y - 35.98*x,
(3, "q0", "q1", "v"): 54.4 + 54.4*x0,
# ("q0", "q2", "a", "c"): -7.66 - 43.64*y - 35.98*x,
("q1", "q1", "c"): 66.86 + 0.3*y + 115.53*x,
("q0", "q0", "cf"): 72.85 + 321.09*y + 69.26*x,
("q2", "q2", "a", "cf"): 51.55 - 26.39*y - 77.95*x
}

U = [-0.25, -0.2, -0.15, -0.1, -0.05, 0.0, 0.05, 0.1, 0.15, 0.2, 0.25] # control inputs

# --- parameters ---
gamma, lamda, delta, tau1, tau2, tau3, rho = 0.85, 100, 0.5, 0.2, 0.06, 0.5, 1/len(U) 

def f(x, w):
    xn = x + w
    xn_clipped = If(xn<0, x, xn)
    return If(x == 0, 0, xn_clipped)


S1 = {
    "q0": [("b", "q1"), ("a", "q2"), ("c", "q0")],
    "q1": [("a", "q1"), ("b", "q1"), ("c", "q1")],
    "q2": [("a", "q2"), ("b", "q2"), ("c", "q2")]
}

pri = {"q0": 3, "q1": 2, "q2": 3}

wp = IntVal(1)   # w = +1
wm = IntVal(-1)  # w = -1

########################################################

odd_pri = [v for v in pri.values() if v % 2 == 1] # odd priorities

def to_z3(e): # convert SCCs to z3 liftable expressions
    return e if is_expr(e) else RealVal(e)

# --- Constraints on domains ---
s = Solver()

for q in S1.keys():
    for (lab, qn) in S1[q]:
        # cond 1
        key1 = (q,qn,lab,"c")
        if key1 in D:
            C1 = to_z3(D[key1])
            C_plus1, C_minus1  = substitute(C1, (y, f(x, wp))), substitute(C1, (y, f(x, wm)))
            E_C1 = sum(rho*((RealVal(0.5)+U[i])*C_plus1 + (RealVal(0.5)-U[i])*C_minus1) for i in range(len(U)))
            s.add(ForAll(x, Implies(x >= 0, E_C1 <= gamma)))
        
        for u in U:
            key2to4 = (q,qn,lab,"c")
            if key2to4 in D:
                C2to4 = to_z3(D[key2to4])
                C_plus2to4, C_minus2to4  = substitute(C2to4, (y, f(x, wp))), substitute(C2to4, (y, f(x, wm)))
                E_C2to4 = (RealVal(0.5)+u)*C_plus2to4 + (RealVal(0.5)-u)*C_minus2to4
                # cond 2
                key2 = (q,qn,lab,"cf")
                if key2 in D:
                    Cf2 = to_z3(D[key2])
                    C_plus2, C_minus2  = substitute(Cf2, (y, f(x, wp))), substitute(Cf2, (y, f(x, wm)))
                    E_Cf2 = (RealVal(0.5)+u)*C_plus2 + (RealVal(0.5)-u)*C_minus2
                    if pri[q] <= pri[qn]:
                        s.add(ForAll(x, Implies(x >= 0, E_Cf2 <= gamma + tau3 * (lamda - E_C2to4))))
    
                for p in S1.keys():
                    # cond 3
                    key3, key3e = (p,q,"cf"), (p,qn,"cf")
                    if key3 not in D or key3e not in D:
                        continue   # skip this iteration
                    Cf3, Cf3e = to_z3(D[key3]), to_z3(D[key3e]) ######
                    s.add(ForAll([x, y], Implies(And(x >= 0, y >= 0), Cf3 >= 0)))
                    if pri[p] <= pri[q] and pri[p] <= pri[qn]:
                        Cf_plus3  = substitute(Cf3, [(y, f(y, wp))])
                        Cf_minus3 = substitute(Cf3, [(y, f(y, wm))])
                        E_Cf3 = (RealVal(0.5)+u)*Cf_plus3 + (RealVal(0.5)-u)*Cf_minus3
                        s.add(ForAll([x, y], Implies(And(x >= 0, y >= 0), E_Cf3 <= Cf3 + tau3 * (lamda - E_C2to4))))
    
                    # cond 4
                    key4, key4e = (p,q,"c"), (p,qn,"c")
                    if key4 not in D or key4e not in D:
                        continue   # skip this iteration
                    C4, C4e = to_z3(D[key4]), to_z3(D[key4e]) ########
                    s.add(ForAll([x, y], Implies(And(x >= 0, y >= 0), C4 >= 0)))
                    C_plus4  = substitute(C4e, [(y, f(y, wp))])
                    C_minus4 = substitute(C4e, [(y, f(y, wm))])
                    E_C4 = (RealVal(0.5)+u)*C_plus4 + (RealVal(0.5)-u)*C_minus4
                    s.add(ForAll([x, y], Implies(And(x >= 0, y >= 0), E_C4 <= C4 + tau3 * (lamda - E_C2to4))))
    
                    # cond 5
                    for l in odd_pri:
                        key5a, key5b = (l,"q0",p,"v"), (l,"q0",q,"v")
                        if key5a in D and key5b in D:
                            V5a = D[key5a]
                            V5b = D[key5b]
                            if l <= pri[p] and pri[p] <= pri[q]:
                                # pass
                                s.add(ForAll([x0, x, y], Implies(And(x0 == 10, x > 0, y > 0), V5a >= 0)))
                                s.add(ForAll([x0, x, y], Implies(And(x0 == 10, x > 0, y > 0), V5b >= 0)))
                                s.add(V5b <= V5a - delta + tau1*(C4 - lamda) + tau2*(Cf3 - lamda))

start_solve = time.time()
result = s.check()
print("sat" if result == sat else "unsat")
end_solve = time.time()
print("Solve time:", end_solve - start_solve)

sat
Solve time: 0.012175798416137695
